In [ ]:
import numpy as np

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip '/content/drive/MyDrive/plates' -d /content/dataset

Archive:  /content/drive/MyDrive/plates.zip
   creating: /content/dataset/plates/
  inflating: /content/dataset/plates/38821295.jpeg  
  inflating: /content/dataset/__MACOSX/plates/._38821295.jpeg  
  inflating: /content/dataset/plates/43231270.jpeg  
  inflating: /content/dataset/__MACOSX/plates/._43231270.jpeg  
  inflating: /content/dataset/plates/31262733.jpeg  
  inflating: /content/dataset/__MACOSX/plates/._31262733.jpeg  
  inflating: /content/dataset/plates/40940583.jpeg  
  inflating: /content/dataset/__MACOSX/plates/._40940583.jpeg  
  inflating: /content/dataset/plates/33565429.jpeg  
  inflating: /content/dataset/__MACOSX/plates/._33565429.jpeg  
  inflating: /content/dataset/plates/36134789.jpeg  
  inflating: /content/dataset/__MACOSX/plates/._36134789.jpeg  
  inflating: /content/dataset/plates/33669827.jpg  
  inflating: /content/dataset/__MACOSX/plates/._33669827.jpg  
  inflating: /content/dataset/plates/38912101.jpeg  
  inflating: /content/dataset/__MACOSX/plates/._

In [ ]:
import os

folder = '/content/dataset/plates'

for name in os.listdir(folder):
    if name.lower().endswith(".jpeg"):
        old = os.path.join(folder, name)
        new = os.path.join(folder, name[:-5] + ".jpg")
        os.rename(old, new)

print("Done!")


Done!


In [ ]:
files = os.listdir(folder)

jpg_files = [f for f in files if f.lower().endswith(".jpg")]

names_no_ext = [f[:-4] for f in jpg_files]   # срезает последние 4 символа

ids = [int(name) for name in names_no_ext]

print(ids)


[32551085, 32530522, 35101217, 41192342, 35271186, 31896264, 40893426, 33603896, 36788031, 33102963, 33702608, 33571524, 39092382, 34935566, 41228235, 34823194, 33696122, 32564123, 35780990, 41094713, 33574518, 40887580, 41452178, 33301460, 43005583, 36494131, 34951293, 35811079, 36506211, 41099527, 32580092, 36588597, 41152856, 33074336, 38984035, 35677260, 43040365, 35880755, 40594155, 43270837, 40573928, 35656107, 30477020, 36367767, 41363303, 33125302, 32594624, 31008134, 33109127, 30399078, 34925963, 36094623, 36113795, 41144718, 38757319, 43158807, 41141248, 41228325, 35223660, 31889613, 41105169, 33675469, 41525122, 36470759, 40938580, 32699845, 30412473, 41256720, 30445483, 41334752, 35667481, 31393799, 31111876, 33323550, 32415155, 31317118, 35080773, 41624372, 43152608, 30464595, 43232083, 31750074, 33595533, 35573235, 32679451, 34935474, 43158673, 41154455, 36497132, 36706831, 41280657, 40538750, 40889671, 36624686, 33674973, 43224556, 30421769, 30449143, 32445501, 35173147,

In [ ]:


import torch

ALPHABET = "0123456789ABCEHIKMOPTXYZ_"
BLANK_IDX = 0  # CTC blank
NUM_CLASSES = len(ALPHABET) + 1  # + blank


char_to_idx = {c: i + 1 for i, c in enumerate(ALPHABET)}  # 1..N
idx_to_char = {i + 1: c for i, c in enumerate(ALPHABET)}


FALLBACK_CHAR = "_"
FALLBACK_IDX = char_to_idx[FALLBACK_CHAR]

def encode_plate(text: str) -> torch.Tensor:
    """
    'AA3165TB' -> tensor([...]) (индексы для CTC)
    """
    text = text.strip().upper()
    ids = [char_to_idx.get(c, FALLBACK_IDX) for c in text]
    return torch.tensor(ids, dtype=torch.long)

def decode_plate(indices):
    """
    Раскодировка последовательности без CTC-логики (просто по индексам).
    Для инференса используем отдельный greedy-decode с CTC, см. ниже.
    """
    chars = []
    for idx in indices:
        if idx == BLANK_IDX:
            continue
        chars.append(idx_to_char.get(int(idx), FALLBACK_CHAR))
    return "".join(chars)


In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/vehicles.csv')
print("CSV file 'vehicles.csv' loaded successfully into DataFrame 'df'.")
df.head()

CSV file 'vehicles.csv' loaded successfully into DataFrame 'df'.


,id,make_name,model_name,year,color_name,source_url,number,image_url
0,36783593,kia,ceed,2010,unknown,https://auto.ria.com/auto_kia_ceed_37709293.html,AA3165TB,https://img.youtube.com/vi/el9-Yu3RegA/default...
1,43225406,bmw,3 series,2010,white,https://auto.ria.com/auto_bmw_3_series_3905574...,AM9193HI,https://cdn4.riastatic.com/photosnew/auto/phot...
2,30530882,lexus,nx 300h,2016,unknown,https://auto.ria.com/auto_lexus_nx_36299469.html,AP9494EI,https://cdn4.riastatic.com/photosnew/auto/phot...
3,35780946,nissan,x-trail,2003,unknown,https://auto.ria.com/auto_nissan_x_trail_37426...,CB9195EK,https://cdn3.riastatic.com/photosnew/auto/phot...
4,40834488,mitsubishi,outlander,2019,unknown,https://auto.ria.com/auto_mitsubishi_outlander...,CE5363CM,https://cdn1.riastatic.com/photosnew/auto/phot...


In [ ]:
df = df[df["id"].isin(ids)]

In [ ]:
from sklearn.model_selection import train_test_split

# Split df into training (70%) and temporary (30%) sets
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42)

# Split the temporary set into validation (50% of temp, 15% of original) and test (50% of temp, 15% of original) sets
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(f"Training set size: {len(train_df)}")
print(f"Validation set size: {len(val_df)}")
print(f"Test set size: {len(test_df)}")
print("DataFrame split into train_df, val_df, and test_df successfully.")

Training set size: 1507
Validation set size: 323
Test set size: 324
DataFrame split into train_df, val_df, and test_df successfully.


In [ ]:
train_df.to_csv('train.csv', index=False)
val_df.to_csv('val.csv', index=False)
test_df.to_csv('test.csv', index=False)


In [ ]:


import torch
import torch.nn as nn



class CRNN(nn.Module):
    def __init__(self, img_height=128, num_channels=3, hidden_size=256):
        super().__init__()

        self.cnn = nn.Sequential(
            # 128x128
            nn.Conv2d(num_channels, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            nn.MaxPool2d(2, 2),   # 64x64

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            nn.MaxPool2d(2, 2),   # 32x32

            nn.Conv2d(128, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(True),

            nn.Conv2d(256, 256, 3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            nn.MaxPool2d((2, 1), (2, 1)),  # 16x32

            nn.Conv2d(256, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(True),

            nn.Conv2d(512, 512, 3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(True),
            nn.MaxPool2d((2, 1), (2, 1)),  # 8x32

            nn.Conv2d(512, 512, 2),        # 7x31 (примерно)
            nn.BatchNorm2d(512),
            nn.ReLU(True),
        )

        self.hidden_size = hidden_size

        # Посчитаем размер входа в LSTM динамически
        with torch.no_grad():
            dummy = torch.zeros(1, num_channels, img_height, img_height)  # 128x128
            conv_out = self.cnn(dummy)  # (1, C, H', W')
            _, c, h, w = conv_out.size()
            self.conv_out_c = c
            self.conv_out_h = h
            self.conv_out_w = w
            rnn_input_size = c * h  # по ширине будем разворачивать в T

        self.rnn = nn.LSTM(
            input_size=rnn_input_size,
            hidden_size=hidden_size,
            num_layers=2,
            bidirectional=True,
            batch_first=False
        )

        self.fc = nn.Linear(hidden_size * 2, NUM_CLASSES)

    def forward(self, x):
        """
        x: (B, C, H, W=128)
        return: (T, B, NUM_CLASSES) — логиты для CTC
        """
        conv = self.cnn(x)          # (B, C, H', W')
        B, C, H, W = conv.size()    # ожидаем H=self.conv_out_h, W=какое-то T

        # делаем последовательность по оси ширины
        conv = conv.permute(3, 0, 1, 2)         # (W', B, C, H')
        conv = conv.contiguous().view(W, B, C*H)  # (T=W', B, C*H')

        rnn_out, _ = self.rnn(conv)            # (T, B, 2*hidden)
        logits = self.fc(rnn_out)              # (T, B, NUM_CLASSES)
        return logits


In [ ]:
import os
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset
import torchvision.transforms as T
import cv2

IMG_ROOT = "/content/dataset/plates"

class PlateDataset(Dataset):
    def __init__(
        self,
        csv_path,
        img_root,
        id_col="id",
        number_col="number",
        color_mode="hls",   # 'rgb', 'hls', 'gray', 'gray_max'
    ):
        self.df = pd.read_csv(csv_path)
        self.img_root = img_root
        self.id_col = id_col
        self.number_col = number_col
        self.color_mode = color_mode

        # трансформ под разные режимы
        #if self.color_mode in ("rgb", "hls"):
            # 3 канала
        self.transform = T.Compose([
            T.Resize((128, 128)),
            T.ToTensor(),
            T.Normalize(
                mean=[0.5, 0.5, 0.5],
                std=[0.5, 0.5, 0.5]
            )
        ])
        # else:
        # 1 канал (gray / gray_max)
        # self.transform = T.Compose([
        # T.Resize((128, 128)),
        # T.ToTensor(),          # (1,H,W)
        # T.Normalize(
        #         mean=[0.5],
        #         std=[0.5]
        #     )
        # ])

    def __len__(self):
        return len(self.df)

    def _get_img_path(self, row):
        img_id = str(row[self.id_col])
        filename = f"{img_id}.jpg"
        path = os.path.join(IMG_ROOT, filename)
        return path

    def _apply_color_mode(self, img: Image.Image) -> Image.Image:
        """img: RGB PIL -> возвращаем PIL в нужном цветовом пространстве."""
        # if self.color_mode == "rgb":
        # return img

        arr = np.array(img)  # RGB, uint8, shape (H,W,3)

        # if self.color_mode == "gray":
        #    gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
        #    return Image.fromarray(gray)

        # if self.color_mode == "gray_max":
        #     gray_max = arr.max(axis=2).astype(np.uint8)
        #     return Image.fromarray(gray_max)

        # OpenCV: RGB -> BGR -> HLS
        bgr = cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)
        hls = cv2.cvtColor(bgr, cv2.COLOR_BGR2HLS)
        # Важно: мы просто храним H,L,S как 3-канальное "изображение"
        # Модели всё равно, как они называются.
        return Image.fromarray(hls)

        # на всякий случай
        return img

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        img_path = self._get_img_path(row)
        plate = str(row[self.number_col])

        img = Image.open(img_path).convert("RGB")
        img = self._apply_color_mode(img)
        img = self.transform(img)

        label = encode_plate(plate)

        return img, label


def collate_fn(batch):
    """
    batch: список (img, label)
    Возвращаем:
    - images: (B, C, H, W)
    - labels_concat: (sum_len,)
    - label_lengths: (B,)
    """
    imgs, labels = zip(*batch)
    imgs = torch.stack(imgs, dim=0)

    label_lengths = torch.tensor([len(l) for l in labels], dtype=torch.long)
    labels_concat = torch.cat(labels, dim=0)

    return imgs, labels_concat, label_lengths


In [ ]:
# train.py

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch import nn, optim


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_csv = "train.csv"
val_csv = "val.csv"

train_dataset = PlateDataset(train_csv, img_root="train_images")
val_dataset = PlateDataset(val_csv, img_root="val_images")

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    collate_fn=collate_fn
)

model = CRNN(img_height=128, num_channels=3, hidden_size=256).to(device)

criterion = nn.CTCLoss(
    blank=BLANK_IDX,
    zero_infinity=True
)

optimizer = optim.Adam(model.parameters(), lr=1e-3)

def train_one_epoch(epoch):
    model.train()
    total_loss = 0.0

    for batch_idx, (images, labels, label_lengths) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)
        label_lengths = label_lengths.to(device)

        logits = model(images)               # (T,B,C)
        log_probs = F.log_softmax(logits, dim=2)

        T, B, C = log_probs.size()
        input_lengths = torch.full(
            size=(B,),
            fill_value=T,
            dtype=torch.long,
            device=device
        )

        loss = criterion(log_probs, labels, input_lengths, label_lengths)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if (batch_idx + 1) % 50 == 0:
            print(f"Epoch {epoch} | Batch {batch_idx+1}/{len(train_loader)} | Loss: {loss.item():.4f}")

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch} TRAIN loss: {avg_loss:.4f}")
    return avg_loss


def validate(epoch):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for images, labels, label_lengths in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            label_lengths = label_lengths.to(device)

            logits = model(images)
            log_probs = F.log_softmax(logits, dim=2)

            T, B, C = log_probs.size()
            input_lengths = torch.full(
                size=(B,),
                fill_value=T,
                dtype=torch.long,
                device=device
            )

            loss = criterion(log_probs, labels, input_lengths, label_lengths)
            total_loss += loss.item()

    avg_loss = total_loss / len(val_loader)
    print(f"Epoch {epoch} VAL loss: {avg_loss:.4f}")
    return avg_loss




In [ ]:

num_epochs = 60

for epoch in range(1, num_epochs + 1):
    train_one_epoch(epoch)
    validate(epoch)
    # Сохранение чекпоинта
    torch.save(model.state_dict(), f"crnn_epoch_{epoch}.pth")

Epoch 1 TRAIN loss: 3.4950
Epoch 1 VAL loss: 3.1375
Epoch 2 TRAIN loss: 2.9273
Epoch 2 VAL loss: 2.7237
Epoch 3 TRAIN loss: 2.6299
Epoch 3 VAL loss: 2.5778
Epoch 4 TRAIN loss: 2.5676
Epoch 4 VAL loss: 2.5617
Epoch 5 TRAIN loss: 2.5406
Epoch 5 VAL loss: 2.5682
Epoch 6 TRAIN loss: 2.5317
Epoch 6 VAL loss: 2.5410
Epoch 7 TRAIN loss: 2.5290
Epoch 7 VAL loss: 2.5469
Epoch 8 TRAIN loss: 2.5280
Epoch 8 VAL loss: 2.5419
Epoch 9 TRAIN loss: 2.5236
Epoch 9 VAL loss: 2.5262
Epoch 10 TRAIN loss: 2.5154
Epoch 10 VAL loss: 2.5295
Epoch 11 TRAIN loss: 2.5162
Epoch 11 VAL loss: 2.5216
Epoch 12 TRAIN loss: 2.5115
Epoch 12 VAL loss: 2.5142
Epoch 13 TRAIN loss: 2.5115
Epoch 13 VAL loss: 2.5707
Epoch 14 TRAIN loss: 2.5056
Epoch 14 VAL loss: 2.5154
Epoch 15 TRAIN loss: 2.4973
Epoch 15 VAL loss: 2.5168
Epoch 16 TRAIN loss: 2.4977
Epoch 16 VAL loss: 2.5100
Epoch 17 TRAIN loss: 2.4943
Epoch 17 VAL loss: 2.4916
Epoch 18 TRAIN loss: 2.4620
Epoch 18 VAL loss: 2.4900
Epoch 19 TRAIN loss: 2.4394
Epoch 19 VAL loss:

In [ ]:
# infer.py

import torch
import torch.nn.functional as F
from PIL import Image
import torchvision.transforms as T


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def ctc_greedy_decode(logits):
    """
    logits: (T, B, C)
    возвращает список строк длины B
    """
    log_probs = F.log_softmax(logits, dim=2)
    max_indices = log_probs.argmax(dim=2)  # (T,B)

    T, B = max_indices.shape
    results = []
    for b in range(B):
        prev = BLANK_IDX
        chars = []
        for t in range(T):
            idx = int(max_indices[t, b])
            if idx != BLANK_IDX and idx != prev:
                chars.append(idx_to_char.get(idx, "?"))
            prev = idx
        results.append("".join(chars))
    return results


def preprocess_image(path):
    transform = T.Compose([
        T.Resize((128, 128)),
        T.ToTensor(),
        T.Normalize(mean=[0.5, 0.5, 0.5],
                    std=[0.5, 0.5, 0.5]),
    ])
    img = Image.open(path).convert("RGB")
    tensor = transform(img).unsqueeze(0)  # (1,C,H,W)
    return tensor


def preprocess_image_hls(path):
    transform = T.Compose([
        T.Resize((128, 128)),
        T.ToTensor(),
        T.Normalize(
            mean=[0.5, 0.5, 0.5],
            std=[0.5, 0.5, 0.5],
        ),
    ])

    # 1) читаем как RGB (PIL)
    img = Image.open(path).convert("RGB")

    # 2) PIL -> numpy (RGB)
    arr = np.array(img)

    # 3) RGB -> BGR -> HLS (OpenCV работает в BGR)
    bgr = cv2.cvtColor(arr, cv2.COLOR_RGB2BGR)
    hls = cv2.cvtColor(bgr, cv2.COLOR_BGR2HLS)

    # 4) обратно в PIL, но уже HLS-картинка (3 канала)
    img_hls = Image.fromarray(hls)

    # 5) дальше обычный transform
    tensor = transform(img_hls).unsqueeze(0)  # (1,3,128,128)
    return tensor

if __name__ == "__main__":
    model = CRNN(img_height=128, num_channels=3, hidden_size=256).to(device)
    model.load_state_dict(torch.load("crnn_epoch_60.pth", map_location=device))
    model.eval()

    img_path = "dataset/plates/30230027.jpg"

    with torch.no_grad():
        x = preprocess_image_hls(img_path).to(device)
        logits = model(x)          # (T,1,C)
        text = ctc_greedy_decode(logits)[0]
        print("Predicted plate:", text)


Predicted plate: KA0315HH


In [ ]:
test_csv = "test.csv"

test_dataset = PlateDataset(test_csv, img_root="test_images")

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4,
    collate_fn=collate_fn
)


In [ ]:
def labels_to_strings(labels, label_lengths):
    """
    labels: 1D tensor с конкатенированными таргетами (CTC),
    label_lengths: длины последовательностей в батче

    возвращает список строк длины B
    """
    texts = []
    idx = 0
    for L in label_lengths:
        L = int(L)
        seq = labels[idx: idx + L]
        text = "".join(idx_to_char[int(c)] for c in seq)
        texts.append(text)
        idx += L
    return texts

In [ ]:

def edit_distance(a: str, b: str) -> int:
    """
    Классический Левенштейн, для CER.
    """
    n, m = len(a), len(b)
    if n == 0:
        return m
    if m == 0:
        return n

    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        dp[i][0] = i
    for j in range(m + 1):
        dp[0][j] = j

    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            dp[i][j] = min(
                dp[i - 1][j] + 1,      # удаление
                dp[i][j - 1] + 1,      # вставка
                dp[i - 1][j - 1] + cost  # замена
            )
    return dp[n][m]

In [ ]:
def evaluate_test():
    model.eval()
    total_loss = 0.0

    total_samples = 0
    exact_matches = 0  # sequence accuracy

    total_chars = 0
    correct_chars = 0  # character accuracy

    total_cer = 0.0    # character error rate

    with torch.no_grad():
        for images, labels, label_lengths in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            label_lengths = label_lengths.to(device)

            logits = model(images)                 # (T,B,C)
            log_probs = F.log_softmax(logits, dim=2)

            T_, B, C = log_probs.size()
            input_lengths = torch.full(
                size=(B,),
                fill_value=T_,
                dtype=torch.long,
                device=device
            )

            loss = criterion(log_probs, labels, input_lengths, label_lengths)
            total_loss += loss.item()

            # ---- Декодируем предсказания и таргеты ----
            preds = ctc_greedy_decode(logits.cpu())  # список строк длины B
            gt_texts = labels_to_strings(labels.cpu(), label_lengths.cpu())
            for pred, gt in zip(preds, gt_texts):
                total_samples += 1

                # sequence accuracy
                print(pred, gt, pred == gt)
                if pred == gt:
                    exact_matches += 1

                # character accuracy
                total_chars += len(gt)
                for p_ch, g_ch in zip(pred, gt):
                    if p_ch == g_ch:
                        correct_chars += 1

                # CER
                if len(gt) > 0:
                    dist = edit_distance(pred, gt)
                    total_cer += dist / len(gt)

    avg_loss = total_loss / len(test_loader)
    seq_acc = exact_matches / total_samples if total_samples > 0 else 0.0
    char_acc = correct_chars / total_chars if total_chars > 0 else 0.0
    cer = total_cer / total_samples if total_samples > 0 else 0.0

    print(f"\n=== TEST METRICS ===")
    print(f"TEST loss: {avg_loss:.4f}")
    print(f"Sequence accuracy: {seq_acc * 100:.2f}%")
    print(f"Character accuracy: {char_acc * 100:.2f}%")
    print(f"Character Error Rate (CER): {cer * 100:.2f}%")

    return {
        "loss": avg_loss,
        "seq_acc": seq_acc,
        "char_acc": char_acc,
        "cer": cer,
    }


In [ ]:
evaluate_test()

AE6934CO AE6934CO True
BC2212BM BC2212BM True
AO1044HO AO1044HO True
CE20020MO AE2002HO False
BC9867PH BC9867PH True
KA9300AX KA9300AX True
BK4968IO BK4968IO True
AX2982OP AX2982OP True
AI7983PT AI7983PT True
AB5451KX AB5451KX True
CB0009E CB0009CB False
BC6961TM BC6961TM True
BM4216ET BA4216ET False
AC0002YA AC0002YA True
AB3276KP AB3276KP True
AE2182OA AE2182OA True
BM8373EC BM8373EC True
KA0806KE KA0806KE True
CE0475YA CE0475YA True
AB1935KT AB1935KT True
AA9597HC AA9597HC True
BH1493MP BH1493MP True
BC0787BP BC0787BP True
AO7468HI AO7468HI True
AX846XC KA0214II False
KE33EC CB7384EI False
AP356EI BX3933CI False
KA6892MK KA6892MK True
CA1490YA CA1490YA True
KE9662AI KE9662AI True
AO6442BK AO6442BK True
BI549BP AI5498OH False
KA9697BB KA9697BB True
AO0770IE AO0770IE True
BX31ET BK3411IE False
AC8000HE AC8000HE True
AX9545AI AX9545KI False
KA3472HK KA3472HK True
KA3524OX KA3524OX True
CB3600EM CB3600EM True
BT8022CT BT8022CT True
BC2000P BC2900TB False
AE0770IH AE0770IH True
A63577KX 

{'loss': 0.4710240892388604,
 'seq_acc': 0.7438271604938271,
 'char_acc': 0.8904320987654321,
 'cer': 0.08796296296296297}

In [ ]:
model.eval()


torch.save(model, "model_full.pt")